## LangChain - Ollama

Run inference on MacOS with Ollama

https://python.langchain.com/docs/how_to/local_llms/

ollama pull llama3.1:8b

When the app is running, all models are automatically served on localhost:11434


In [1]:
conda env list


# conda environments:
#
base                   /opt/anaconda3
HuggingFaceBook        /opt/anaconda3/envs/HuggingFaceBook
islp                   /opt/anaconda3/envs/islp
langsmith            * /opt/anaconda3/envs/langsmith
py313                  /opt/anaconda3/envs/py313
python-ds-ml-bootcamp   /opt/anaconda3/envs/python-ds-ml-bootcamp


Note: you may need to restart the kernel to use updated packages.


In [1]:
pip install -qU langchain_ollama

Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_ollama import OllamaLLM
llm = OllamaLLM(model="llama3.1:8b")
llm.invoke("The first man on the moon was ...")

ModuleNotFoundError: No module named 'langchain_ollama'

In [3]:
for chunk in llm.stream("The first man on the moon was ..."):
    print(chunk, end="|", flush=True)

...| Neil| Armstrong|!| He| stepped| out| of| the| lunar| module| Eagle| and| onto| the| surface| of| the| Moon| on| July| |20|,| |196|9|,| famously| declaring| "|That|'s| one| small| step| for| man|,| one| giant| leap| for| mankind|."||

In [4]:
from langchain_ollama import ChatOllama
chat_model = ChatOllama(model="llama3.1:8b")
chat_model.invoke("Who was the first man on the moon?")

AIMessage(content='The answer is Neil Armstrong. On July 20, 1969, during the Apollo 11 mission, Armstrong became the first person to set foot on the lunar surface. He famously declared, "That\'s one small step for man, one giant leap for mankind," as he stepped off the lunar module Eagle onto the moon\'s surface.\n\nHowever, it\'s worth noting that Armstrong was followed by Edwin "Buzz" Aldrin, who also walked on the moon during the same mission. They spent a total of two and a half hours outside the lunar module, collecting samples and conducting experiments.\n\nArmstrong died in 2012 at the age of 82, but his legacy lives on as an iconic figure in space exploration history!', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2025-08-09T16:55:34.723285Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2232540583, 'load_duration': 35541958, 'prompt_eval_count': 19, 'prompt_eval_duration': 186266375, 'eval_count': 147, 'eval_duration': 2010339417, 

In [8]:
# !source ~/.zshrc

In [1]:
import os
from langchain import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEndpoint
from langchain_core.runnables.history import RunnableWithMessageHistory
import jk_secrets

os.environ['HUGGINGFACEHUB_API_TOKEN'] = jk_secrets.HUGGINGFACEHUB_API_TOKEN

template = '''
Question: {question}
Answer: 
'''

prompt = PromptTemplate(
    template = template,
    input_variables = ['question']
)

hub_llm = HuggingFaceEndpoint(
    endpoint_url="https://api-inference.huggingface.co/models/HuggingFaceH4/zephyr-7b-alpha", 
    temperature = 1
)

class SessionHistory:
    def __init__(self):
        self.messages = []

    def add_messages(self, messages):
        self.messages.extend(messages)

    def get_messages(self):
        return self.messages

session_history = SessionHistory()

def get_session_history():
    return session_history

llm_chain = RunnableWithMessageHistory(
    prompt | hub_llm | StrOutputParser(),
    get_session_history = get_session_history
)

while True:
    # get the user input
    user_question = input("Ask a question (type 'exit' to stop): ")

    # exit condition
    if user_question.lower() == "quit":
        print("Ending conversation.")
        break

    # prepare the input data
    input_data = {"question": user_question}

    response = llm_chain.invoke(input_data)

    session_history.add_messages([
        {"role": "user", "content": user_question},
        {"role": "assistant", "content": response}
    ])

    # display the response
    print(f"AI: {response}")


ModuleNotFoundError: No module named 'langchain_huggingface'